# Exportar y Preprocesar Datos de SQL Server a CSV y Parquet con Polars

Este notebook realiza la extracción, preprocesamiento y preparación para minería de datos (Weka) de forma rápida, eficiente y en memoria con **Polars**.

Integra la lógica de preprocesamiento de datos y la preparación de Weka:
1. **Idempotente y no destructivo:** No altera la tabla original en SQL Server.
2. **Seguro:** Funciona incluso con permisos de solo lectura.
3. **Optimizado:** Todo el preprocesamiento pesado y la manipulación de nulos se realiza en memoria usando Polars.
4. **Control de Origen Flexible:** Permite elegir entre conectar a SQL Server o leer un archivo `.parquet` local.
5. **Compatibilidad con Weka:** Exporta el CSV en codificación `Windows-1252` con caracteres especiales intactos (evitando errores de lectura en Weka en sistemas Windows) y con la variable clase `HOSPITALIZADO` al final.

## 1. Configuración de Variables

In [10]:
# === CONFIGURA TUS VARIABLES DE CONEXIÓN Y DATOS AQUÍ ===
USAR_PARQUET_LOCAL = True            # True para cargar datos desde un archivo Parquet local, False para conectar a SQL Server
RUTA_PARQUET_LOCAL = 'parquets/entidad_21_exportada.parquet' # Ruta del archivo Parquet local

SERVER = 'localhost'                  # Nombre o IP del servidor SQL Server
DATABASE = 'KddPuebla'                # Nombre de la base de datos
TABLA = 'entidad_21'                  # Nombre de la tabla que deseas exportar (si USAR_PARQUET_LOCAL=False)

WINDOWS_AUTH = False                  # True si usas Autenticación de Windows
USUARIO = 'sa'                        # Tu usuario (ignorar si WINDOWS_AUTH=True)
CONTRASEÑA = 'admin'                  # Tu contraseña (ignorar si WINDOWS_AUTH=True)

DRIVER = 'ODBC Driver 17 for SQL Server' # Driver instalado en tu sistema
TIMEOUT_CONEXION = 15                 # Limitar tiempo de espera de conexión a 15 segundos
EVITAR_BLOQUEOS = True                  # Agrega 'WITH (NOLOCK)' para no atascarse si la tabla está bloqueada
CARPETA_SALIDA = 'csv_exportados'     # Carpeta de destino para los archivos CSV y Parquet

## 2. Origen de Datos (Carga local de Parquet o Descarga de SQL Server)
Establece la conexión a la base de datos y descarga la tabla completa en un DataFrame de Polars, o carga de forma directa el archivo Parquet local seleccionado.

In [11]:
import polars as pl
from sqlalchemy import create_engine
import urllib.parse
import os

if USAR_PARQUET_LOCAL:
    print(f"[*] Cargando datos desde el archivo Parquet local: '{RUTA_PARQUET_LOCAL}'...")
    try:
        if not os.path.exists(RUTA_PARQUET_LOCAL):
            raise FileNotFoundError(f'El archivo Parquet local no existe en la ruta: {os.path.abspath(RUTA_PARQUET_LOCAL)}')
        df_raw = pl.read_parquet(RUTA_PARQUET_LOCAL)
        print(f'[+] Datos cargados exitosamente: se leyeron {df_raw.height} filas y {df_raw.width} columnas.')
    except Exception as e:
        print(f'\n[!] ERROR AL CARGAR EL PARQUET LOCAL:\n{e}')
        df_raw = None
else:
    # 1. Codificar contraseña en formato URI
    pwd_encoded = urllib.parse.quote_plus(CONTRASEÑA)

    # 2. Construir la URI de conexión para SQLAlchemy
    if WINDOWS_AUTH:
        conn_uri = f'mssql+pyodbc://@{SERVER}/{DATABASE}?driver={DRIVER}&trusted_connection=yes'
    else:
        conn_uri = f'mssql+pyodbc://{USUARIO}:{pwd_encoded}@{SERVER}/{DATABASE}?driver={DRIVER}'

    # 3. Configurar el motor de conexión
    engine = create_engine(
        conn_uri,
        connect_args={'timeout': TIMEOUT_CONEXION}
    )

    # 4. Construir la consulta SQL
    if EVITAR_BLOQUEOS:
        query = f'SELECT * FROM {TABLA} WITH (NOLOCK)'
    else:
        query = f'SELECT * FROM {TABLA}'

    print(f"[*] Iniciando descarga desde SQL Server para la tabla '{TABLA}'...")
    try:
        print(f"[+] Intentando conectar al servidor '{SERVER}' (Límite: {TIMEOUT_CONEXION} segundos)...")
        with engine.connect() as conn:
            print('[+] ¡Conexión establecida correctamente!')
            print(f"[+] Ejecutando consulta: '{query}'")
            print('    (Descargando filas, por favor espera...)')
            df_raw = pl.read_database(query, connection=conn)
            print(f'[+] Descarga finalizada: se leyeron {df_raw.height} filas y {df_raw.width} columnas.')
    except Exception as e:
        print(f'\n[!] ERROR EN CONEXIÓN/DESCARGA:\n{e}')
        df_raw = None

[*] Cargando datos desde el archivo Parquet local: 'parquets/entidad_21_exportada.parquet'...
[+] Datos cargados exitosamente: se leyeron 722660 filas y 46 columnas.


## 3. Preprocesamiento de Datos con Polars
En esta sección se integra la lógica de limpieza y transformación ejecutada de forma secuencial sobre el DataFrame en memoria.

Si el DataFrame cargado ya tiene las columnas preprocesadas (por ejemplo, si se cargó un Parquet preprocesado), el flujo lo detectará automáticamente y mantendrá la estructura existente sin arrojar errores.

In [12]:
# Validación de carga correcta
if 'df_raw' not in locals() or df_raw is None:
    raise ValueError('Error: No se han cargado datos. Verifica la configuración de la conexión a SQL Server o la ruta del archivo Parquet local.')

# Verificar si el DataFrame ya está preprocesado basándonos en la existencia de columnas originales
columnas_originales_requeridas = ['EMBARAZO', 'SEXO', 'TIPO_PACIENTE']
necesita_preprocesamiento = any(col in df_raw.columns for col in columnas_originales_requeridas)

if necesita_preprocesamiento:
    print('[*] Columnas originales detectadas. Iniciando preprocesamiento...')
    df = df_raw.clone()
else:
    print('[+] Los datos ya contienen columnas preprocesadas. Se omitirá el preprocesamiento en Polars.')
    df = df_raw.clone()

[*] Columnas originales detectadas. Iniciando preprocesamiento...


### Paso 1: Tratamiento de Valores Especiales (Lógica de SQLQueryPaso1.sql)
1. Crea banderas de realización de pruebas de laboratorio (`MUESTRA_LAB_TOMADA`, `MUESTRA_ANTIGENO_TOMADA`, `PANEL_VIRAL_REALIZADO`).
2. Limpia valores especiales (`SE IGNORA`, `NO APLICA`, `NO ESPECIFICADO`) convirtiéndolos a `null` (que Weka lee como `?`), excepto en casos biológica o clínicamente justificados (como `EMBARAZO` imputado a `NO` para hombres).

In [13]:
if necesita_preprocesamiento:
    # 1A y 1H-i. Crear columnas de banderas para resultados de laboratorio
    df = df.with_columns([
        pl.when(pl.col('RESULTADO_LAB').is_null() | (pl.col('RESULTADO_LAB') == 'NO APLICA (CASO SIN MUESTRA)'))
          .then(0).otherwise(1).cast(pl.Int8).alias('MUESTRA_LAB_TOMADA'),
        pl.when(pl.col('RESULTADO_ANTIGENO').is_null() | (pl.col('RESULTADO_ANTIGENO') == 'NO APLICA (CASO SIN MUESTRA)'))
          .then(0).otherwise(1).cast(pl.Int8).alias('MUESTRA_ANTIGENO_TOMADA'),
        pl.when(pl.col('RESULTADO_PCR').is_null() | (pl.col('RESULTADO_PCR') == 'NO APLICA (CASO SIN MUESTRA)'))
          .then(0).otherwise(1).cast(pl.Int8).alias('PANEL_VIRAL_REALIZADO')
    ])
    
    # 1B. EMBARAZO (Hombres 'NO APLICA' -> 'NO', 'SE IGNORA' -> NULL)
    df = df.with_columns(
        pl.when(pl.col('EMBARAZO') == 'NO APLICA').then(pl.lit('NO'))
          .when(pl.col('EMBARAZO') == 'SE IGNORA').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('EMBARAZO')).alias('EMBARAZO')
    )
    
    # 1C. Comorbilidades (10 variables, 'SE IGNORA' -> NULL)
    comorbilidades = ['ASMA', 'CARDIOVASCULAR', 'DIABETES', 'EPOC', 'HIPERTENSION', 
                      'INMUSUPR', 'OBESIDAD', 'OTRA_COM', 'RENAL_CRONICA', 'TABAQUISMO']
    df = df.with_columns([
        pl.when(pl.col(c) == 'SE IGNORA').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col(c)).alias(c)
        for c in comorbilidades
    ])
    
    # 1D. NEUMONIA ('NO ESPECIFICADO' -> NULL)
    df = df.with_columns(
        pl.when(pl.col('NEUMONIA') == 'NO ESPECIFICADO').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('NEUMONIA')).alias('NEUMONIA')
    )
    
    # 1E. HABLA_LENGUA_INDIG e INDIGENA ('NO ESPECIFICADO' -> NULL)
    df = df.with_columns([
        pl.when(pl.col('HABLA_LENGUA_INDIG') == 'NO ESPECIFICADO').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('HABLA_LENGUA_INDIG')).alias('HABLA_LENGUA_INDIG'),
        pl.when(pl.col('INDIGENA') == 'NO ESPECIFICADO').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('INDIGENA')).alias('INDIGENA')
    ])
    
    # 1G. Variables categóricas (ENTIDAD_NAC, SECTOR)
    df = df.with_columns([
        pl.when(pl.col('ENTIDAD_NAC') == 'NO ESPECIFICADO').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('ENTIDAD_NAC')).alias('ENTIDAD_NAC'),
        pl.when(pl.col('SECTOR') == 'NO ESPECIFICADO').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('SECTOR')).alias('SECTOR')
    ])
    
    # 1H-ii. Limpiar columnas originales de laboratorio (Casos sin muestra y pendientes -> NULL)
    df = df.with_columns([
        pl.when(pl.col('RESULTADO_LAB') == 'NO APLICA (CASO SIN MUESTRA)').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('RESULTADO_LAB')).alias('RESULTADO_LAB'),
        pl.when(pl.col('RESULTADO_ANTIGENO') == 'NO APLICA (CASO SIN MUESTRA)').then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('RESULTADO_ANTIGENO')).alias('RESULTADO_ANTIGENO'),
        pl.when(pl.col('RESULTADO_PCR').is_in(['NO APLICA (CASO SIN MUESTRA)', 'PENDIENTE'])).then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('RESULTADO_PCR')).alias('RESULTADO_PCR')
    ])
    
    # 1I. MUNICIPIO_RES ('NO ESPECIFICADO', 'SE IGNORA', 'NO APLICA' -> NULL)
    df = df.with_columns(
        pl.when(pl.col('MUNICIPIO_RES').is_in(['NO ESPECIFICADO', 'SE IGNORA', 'NO APLICA'])).then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.col('MUNICIPIO_RES')).alias('MUNICIPIO_RES')
    )
    print('[+] Paso 1: Tratamiento de valores especiales completado de forma no destructiva.')
else:
    print('[+] Paso 1 omitido (los datos ya están preprocesados).')

[+] Paso 1: Tratamiento de valores especiales completado de forma no destructiva.


### Paso 2: Variables Calculadas y Derivadas (Lógica de SQLQueryPaso2.sql)
1. Calcula `TOTAL_COMORBILIDADES` como la suma de las comorbilidades del paciente (0 a 10).
2. Crea `TIENE_COMORBILIDAD` (SI/NO) si tiene al menos una.
3. Segmenta la edad en 5 tramos en `GRUPO_ETARIO` (0-17, 18-39, 40-59, 60-79, 80+) y crea la bandera `ES_ADULTO_MAYOR` (EDAD >= 60).
4. Deriva `PERTENECE_POB_INDIGENA` unificando las dos columnas correlacionadas de origen.
5. Determina si el paciente es residente del estado de Puebla (`ES_RESIDENTE_PUEBLA`) y si fue atendido fuera de su entidad (`ATENDIDO_FUERA_ENTIDAD`).
6. Unifica la clasificación diagnóstica final en `DIAGNOSTICO_FINAL` (combinando `CLASIFICACION_FINAL` y `CLASIFICACION_FINAL_COVID`).

In [14]:
if necesita_preprocesamiento:
    comorbilidades = ['ASMA', 'CARDIOVASCULAR', 'DIABETES', 'EPOC', 'HIPERTENSION', 
                      'INMUSUPR', 'OBESIDAD', 'OTRA_COM', 'RENAL_CRONICA', 'TABAQUISMO']
    
    # 2B. TOTAL_COMORBILIDADES (Suma de comorbilidades activas 'SI')
    df = df.with_columns(
        pl.sum_horizontal([
            (pl.col(c) == 'SI').fill_null(False).cast(pl.Int32)
            for c in comorbilidades
        ]).alias('TOTAL_COMORBILIDADES')
    )
    
    # 2C. TIENE_COMORBILIDAD
    df = df.with_columns(
        pl.when(pl.col('TOTAL_COMORBILIDADES') >= 1).then(pl.lit('SI'))
          .when(pl.col('TOTAL_COMORBILIDADES') == 0).then(pl.lit('NO'))
          .otherwise(pl.lit(None, dtype=pl.String)).alias('TIENE_COMORBILIDAD')
    )
    
    # 2D. GRUPO_ETARIO (Edades > 110 se consideran errores y van a NULL)
    df = df.with_columns(
        pl.when(pl.col('EDAD').is_null() | (pl.col('EDAD') > 110)).then(pl.lit(None, dtype=pl.String))
          .when(pl.col('EDAD') <= 17).then(pl.lit('0-17'))
          .when(pl.col('EDAD') <= 39).then(pl.lit('18-39'))
          .when(pl.col('EDAD') <= 59).then(pl.lit('40-59'))
          .when(pl.col('EDAD') <= 79).then(pl.lit('60-79'))
          .otherwise(pl.lit('80+')).alias('GRUPO_ETARIO')
    )
    
    # 2E. ES_ADULTO_MAYOR
    df = df.with_columns(
        pl.when(pl.col('EDAD').is_null() | (pl.col('EDAD') > 110)).then(pl.lit(None, dtype=pl.String))
          .when(pl.col('EDAD') >= 60).then(pl.lit('SI'))
          .otherwise(pl.lit('NO')).alias('ES_ADULTO_MAYOR')
    )
    
    # 2F. PERTENECE_POB_INDIGENA
    df = df.with_columns(
        pl.when((pl.col('HABLA_LENGUA_INDIG') == 'SI') | (pl.col('INDIGENA') == 'SI')).then(pl.lit('SI'))
          .when(pl.col('HABLA_LENGUA_INDIG').is_null() & pl.col('INDIGENA').is_null()).then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.lit('NO')).alias('PERTENECE_POB_INDIGENA')
    )
    
    # 2G. ES_RESIDENTE_PUEBLA
    df = df.with_columns(
        pl.when(pl.col('ENTIDAD_RES') == 'PUEBLA').then(pl.lit('SI'))
          .when(pl.col('ENTIDAD_RES').is_null()).then(pl.lit(None, dtype=pl.String))
          .otherwise(pl.lit('NO')).alias('ES_RESIDENTE_PUEBLA')
    )
    
    # 2H. ATENDIDO_FUERA_ENTIDAD
    df = df.with_columns(
        pl.when(pl.col('ENTIDAD_UM').is_null() | pl.col('ENTIDAD_RES').is_null()).then(pl.lit(None, dtype=pl.String))
          .when(pl.col('ENTIDAD_UM') != pl.col('ENTIDAD_RES')).then(pl.lit('SI'))
          .otherwise(pl.lit('NO')).alias('ATENDIDO_FUERA_ENTIDAD')
    )
    
    # 2I. DIAGNOSTICO_FINAL
    df = df.with_columns(
        pl.when(pl.col('CLASIFICACION_FINAL').is_not_null()).then(pl.col('CLASIFICACION_FINAL'))
          .when(pl.col('CLASIFICACION_FINAL_COVID').is_not_null()).then(pl.col('CLASIFICACION_FINAL_COVID'))
          .otherwise(pl.lit(None, dtype=pl.String)).alias('DIAGNOSTICO_FINAL')
    )
    print('[+] Paso 2: Cálculo de variables derivadas completado en memoria.')
else:
    print('[+] Paso 2 omitido (los datos ya están preprocesados).')

[+] Paso 2: Cálculo de variables derivadas completado en memoria.


### Paso 3: Conversión a Binario
Codifica todas las variables binarias nominales (valores `SI` y `NO`) a formato numérico `1` y `0` (tipo `Int8` para optimizar), dejando los nulos intactos.
- `HOSPITALIZADO`: Variable objetivo, binaria (1 = HOSPITALIZADO, 0 = AMBULATORIO).
- Variables con sufijo `_BIN`: `SEXO_BIN`, `EMBARAZO_BIN`, comorbilidades, y variables binarias derivadas.

In [15]:
if necesita_preprocesamiento:
    # 3B. Variable objetivo (HOSPITALIZADO)
    df = df.with_columns(
        pl.when(pl.col('TIPO_PACIENTE') == 'HOSPITALIZADO').then(1)
          .when(pl.col('TIPO_PACIENTE') == 'AMBULATORIO').then(0)
          .otherwise(pl.lit(None, dtype=pl.Int8)).alias('HOSPITALIZADO')
    )
    
    # 3C. Grupo A: Columnas originales SI/NO
    df = df.with_columns(
        pl.when(pl.col('SEXO') == 'HOMBRE').then(1)
          .when(pl.col('SEXO') == 'MUJER').then(0)
          .otherwise(pl.lit(None, dtype=pl.Int8)).alias('SEXO_BIN')
    )
    
    df = df.with_columns(
        pl.when(pl.col('ORIGEN') == 'USMER').then(1)
          .when(pl.col('ORIGEN') == 'FUERA DE USMER').then(0)
          .otherwise(pl.lit(None, dtype=pl.Int8)).alias('ORIGEN_BIN')
    )
    
    cols_si_no = [
        'EMBARAZO', 'NEUMONIA', 'DIABETES', 'EPOC', 'ASMA', 'INMUSUPR', 
        'HIPERTENSION', 'OTRA_COM', 'CARDIOVASCULAR', 'OBESIDAD', 'RENAL_CRONICA', 
        'TABAQUISMO', 'TOMA_MUESTRA_LAB', 'TOMA_MUESTRA_ANTIGENO'
    ]
    df = df.with_columns([
        pl.when(pl.col(c) == 'SI').then(1)
          .when(pl.col(c) == 'NO').then(0)
          .otherwise(pl.lit(None, dtype=pl.Int8)).alias(f'{c}_BIN')
        for c in cols_si_no
    ])
    
    # 3D. Grupo B: Variables derivadas SI/NO del Paso 2
    cols_derived_si_no = [
        'TIENE_COMORBILIDAD', 'ES_ADULTO_MAYOR', 'PERTENECE_POB_INDIGENA', 
        'ES_RESIDENTE_PUEBLA', 'ATENDIDO_FUERA_ENTIDAD'
    ]
    df = df.with_columns([
        pl.when(pl.col(c) == 'SI').then(1)
          .when(pl.col(c) == 'NO').then(0)
          .otherwise(pl.lit(None, dtype=pl.Int8)).alias(f'{c}_BIN')
        for c in cols_derived_si_no
    ])
    print('[+] Paso 3: Conversión de variables nominales binarias a numéricas 1/0 completada.')
else:
    print('[+] Paso 3 omitido (los datos ya están preprocesados).')

[+] Paso 3: Conversión de variables nominales binarias a numéricas 1/0 completada.


### Paso 4: Selección de Columnas Finales
Filtra la estructura final del dataset para quedarnos exactamente con las **36 columnas** necesarias para el modelado, eliminando campos redundantes, identificadores, fechas, o variables con fuga de información (como `INTUBADO` y `UCI`).

In [16]:
# Definición de las 36 columnas preprocesadas finales
columnas_finales = [
    'EDAD', 'DIAS_SINTOMAS', 'TOTAL_COMORBILIDADES', 'HOSPITALIZADO',
    'SEXO_BIN', 'EMBARAZO_BIN', 'NEUMONIA_BIN', 'DIABETES_BIN', 'EPOC_BIN',
    'ASMA_BIN', 'INMUSUPR_BIN', 'HIPERTENSION_BIN', 'OTRA_COM_BIN',
    'CARDIOVASCULAR_BIN', 'OBESIDAD_BIN', 'RENAL_CRONICA_BIN', 'TABAQUISMO_BIN',
    'TOMA_MUESTRA_LAB_BIN', 'TOMA_MUESTRA_ANTIGENO_BIN', 'ORIGEN_BIN',
    'TIENE_COMORBILIDAD_BIN', 'ES_ADULTO_MAYOR_BIN', 'PERTENECE_POB_INDIGENA_BIN',
    'ES_RESIDENTE_PUEBLA_BIN', 'ATENDIDO_FUERA_ENTIDAD_BIN',
    'MUESTRA_LAB_TOMADA', 'MUESTRA_ANTIGENO_TOMADA', 'PANEL_VIRAL_REALIZADO',
    'GRUPO_ETARIO', 'SECTOR', 'OTRO_CASO', 'DIAGNOSTICO_FINAL',
    'RESULTADO_LAB', 'RESULTADO_ANTIGENO', 'RESULTADO_PCR', 'CLASIFICACION_FINAL_FLU'
]

# Seleccionar solo las columnas finales
df_final = df.select(columnas_finales)
print(f'[+] Paso 4: Selección completada. El DataFrame tiene {df_final.width} columnas (Esperado: 36).')

[+] Paso 4: Selección completada. El DataFrame tiene 36 columnas (Esperado: 36).


### Paso 5: Preparación de Datos para Weka
Aplica las adaptaciones requeridas para garantizar la correcta compatibilidad y lectura del archivo en **Weka**:
1. Asegura que las columnas binarias y banderas de prueba de laboratorio estén tipadas correctamente como enteros (`Int8`).
2. **Orden de las Columnas:** Mueve la variable objetivo (`HOSPITALIZADO`) a la **última columna** del dataset, que es el formato de convención estándar que espera Weka para identificar automáticamente la variable clase.

In [17]:
print('[*] Preparando DataFrame final para Weka...')

# 1. Asegurar que columnas bool / bit de la base de datos se traten como enteros 1/0
bool_cols = ['MUESTRA_LAB_TOMADA', 'MUESTRA_ANTIGENO_TOMADA', 'PANEL_VIRAL_REALIZADO']
df_weka = df_final.with_columns([
    pl.col(c).cast(pl.Int8) for c in bool_cols if c in df_final.columns
])

# 2. Asegurar que las columnas float con nulls binarias sean tipadas a enteros
float_bin_cols = [
    'EMBARAZO_BIN', 'NEUMONIA_BIN', 'DIABETES_BIN', 'EPOC_BIN',
    'ASMA_BIN', 'INMUSUPR_BIN', 'HIPERTENSION_BIN', 'OTRA_COM_BIN',
    'CARDIOVASCULAR_BIN', 'OBESIDAD_BIN', 'RENAL_CRONICA_BIN',
    'TABAQUISMO_BIN', 'ES_ADULTO_MAYOR_BIN', 'PERTENECE_POB_INDIGENA_BIN'
]
df_weka = df_weka.with_columns([
    pl.col(c).cast(pl.Int8) for c in float_bin_cols if c in df_weka.columns
])

# 3. Reordenar columnas para posicionar la variable objetivo (HOSPITALIZADO) al final
cols_weka = [c for c in df_weka.columns if c != 'HOSPITALIZADO'] + ['HOSPITALIZADO']
df_weka = df_weka.select(cols_weka)

print('[+] Paso 5: Preparación de datos para Weka finalizada exitosamente.')

[*] Preparando DataFrame final para Weka...
[+] Paso 5: Preparación de datos para Weka finalizada exitosamente.


## 4. Verificación de Integridad y Estadísticas
Realiza la comprobación estadística del dataset preparado para Weka para asegurar la consistencia y calidad de los datos.

In [18]:
print('=== REPORTE DE INTEGRIDAD DEL DATASET WEKA ===')
print(f'- Dimensiones finales: {df_weka.height} filas y {df_weka.width} columnas.')

# Confirmación de la dimensión de columnas
if df_weka.width == 36:
    print('[+] ¡Éxito! El dataset cuenta con las 36 columnas esperadas.')
else:
    print(f'[!] Advertencia: Se detectaron {df_weka.width} columnas (se esperaban 36).')

# Verificar que la variable clase está al final
if df_weka.columns[-1] == 'HOSPITALIZADO':
    print('[+] ¡Éxito! La variable objetivo HOSPITALIZADO está al final de las columnas.')
else:
    print(f'[!] Error: La última columna es {df_weka.columns[-1]}, se esperaba HOSPITALIZADO.')

# Distribución de la variable objetivo (HOSPITALIZADO)
print('\nDistribución de HOSPITALIZADO:')
try:
    dist = df_weka.group_by('HOSPITALIZADO').len().sort('HOSPITALIZADO')
    print(dist)
except Exception:
    # Compatibilidad con versiones anteriores de Polars
    dist = df_weka.group_by('HOSPITALIZADO').count().sort('HOSPITALIZADO')
    print(dist)

# Conteo de valores nulos en variables clave
print('\nConteo de nulos en variables clave (quedarán representados como ? en Weka):')
variables_verificacion = ['EMBARAZO_BIN', 'ASMA_BIN', 'NEUMONIA_BIN', 'DIAGNOSTICO_FINAL']
for var in variables_verificacion:
    if var in df_weka.columns:
        nulos = df_weka[var].is_null().sum()
        print(f'  - {var}: {nulos} nulos')

# Verificar que HOSPITALIZADO no tiene nulos
nulos_hosp = df_weka['HOSPITALIZADO'].is_null().sum()
assert nulos_hosp == 0, 'ERROR: HOSPITALIZADO tiene nulos'
print('\n[+] HOSPITALIZADO sin nulls: OK')

=== REPORTE DE INTEGRIDAD DEL DATASET WEKA ===
- Dimensiones finales: 722660 filas y 36 columnas.
[+] ¡Éxito! El dataset cuenta con las 36 columnas esperadas.
[+] ¡Éxito! La variable objetivo HOSPITALIZADO está al final de las columnas.

Distribución de HOSPITALIZADO:
shape: (2, 2)
┌───────────────┬────────┐
│ HOSPITALIZADO ┆ len    │
│ ---           ┆ ---    │
│ i8            ┆ u32    │
╞═══════════════╪════════╡
│ 0             ┆ 621484 │
│ 1             ┆ 101176 │
└───────────────┴────────┘

Conteo de nulos en variables clave (quedarán representados como ? en Weka):
  - EMBARAZO_BIN: 2181 nulos
  - ASMA_BIN: 740 nulos
  - NEUMONIA_BIN: 2765 nulos
  - DIAGNOSTICO_FINAL: 0 nulos

[+] HOSPITALIZADO sin nulls: OK


## 5. Exportación de Resultados
Guarda los archivos en los formatos correspondientes en la carpeta destino:
1. **Parquet estándar** (UTF-8) para uso con Polars/Python.
2. **CSV estándar** (UTF-8).
3. **CSV preparado para Weka**:
   - Codificación **Windows-1252 (CP1252)** para evitar la corrupción de caracteres especiales (acentos y Ñ) que hace fallar el lector nominal de Weka en Windows.
   - Representación de valores faltantes (nulos) como `?`.

In [19]:
# Crear carpeta destino si no existe
os.makedirs(CARPETA_SALIDA, exist_ok=True)

ruta_parquet = os.path.join(CARPETA_SALIDA, f'{TABLA}_exportada.parquet')
ruta_csv = os.path.join(CARPETA_SALIDA, f'{TABLA}_exportada.csv')
ruta_weka = os.path.join(CARPETA_SALIDA, f'{TABLA}_weka.csv')

print(f"[*] Guardando datos en la carpeta '{CARPETA_SALIDA}'...")

# 1. Guardar Parquet estándar (UTF-8)
print(f'    - Escribiendo Parquet estándar: {ruta_parquet}')
df_final.write_parquet(ruta_parquet)

# 2. Guardar CSV estándar (UTF-8)
print(f'    - Escribiendo CSV estándar: {ruta_csv}')
df_final.write_csv(ruta_csv)

# 3. Guardar CSV preparado para Weka (Codificado en Windows-1252, nulos como '?')
print(f'    - Escribiendo CSV para Weka (Windows-1252, nulos como "?"): {ruta_weka}')
csv_str = df_weka.write_csv(null_value='?')
with open(ruta_weka, 'w', encoding='windows-1252', errors='replace') as f:
    f.write(csv_str)

print(f'\n[+] ¡Proceso de exportación completado exitosamente!')
print(f'    - Parquet estándar: {os.path.abspath(ruta_parquet)}')
print(f'    - CSV estándar (UTF-8): {os.path.abspath(ruta_csv)}')
print(f'    - CSV Weka (CP1252): {os.path.abspath(ruta_weka)}')
print('\n¡Listo para cargar directamente en Weka!')

[*] Guardando datos en la carpeta 'csv_exportados'...
    - Escribiendo Parquet estándar: csv_exportados\entidad_21_exportada.parquet
    - Escribiendo CSV estándar: csv_exportados\entidad_21_exportada.csv
    - Escribiendo CSV para Weka (Windows-1252, nulos como "?"): csv_exportados\entidad_21_weka.csv

[+] ¡Proceso de exportación completado exitosamente!
    - Parquet estándar: d:\Users\JoseLuis\Desktop\UTP\Extracción de conocimiento db\Polars\csv_exportados\entidad_21_exportada.parquet
    - CSV estándar (UTF-8): d:\Users\JoseLuis\Desktop\UTP\Extracción de conocimiento db\Polars\csv_exportados\entidad_21_exportada.csv
    - CSV Weka (CP1252): d:\Users\JoseLuis\Desktop\UTP\Extracción de conocimiento db\Polars\csv_exportados\entidad_21_weka.csv

¡Listo para cargar directamente en Weka!
